<a href="https://colab.research.google.com/github/Lezier/phenological_prediction_notebook/blob/main/phenological_prediction_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicción fenológica reproducible en Google Colab

Port público de `phenological_prediction` **0.1.0-rc.3**, commit fuente
`253358e75ac6bf72333d358251613473f59961ee`.

El flujo prepara A/A′/B, compara una red densa con Random Forest usando los
mismos folds y pesos por fold, registra dispersión y tiempos, entrena Random
Forest A y ejecuta una inferencia con siete variables. Los CSV se descargan
desde este repositorio y se aceptan únicamente si coinciden con sus SHA-256.

> Uso experimental con datos europeos; no validado para operación en Chile.
> Las probabilidades del prototipo no están calibradas.

Datos: CC BY-NC 4.0 con atribución según `DATA_LICENSE.md`. El código no tiene
licencia abierta; véase `CODE_LICENSE.md`.

## 1. Runtime y dependencias

Se exige CPython 3.13 y un conjunto de versiones fijadas. La celda instala
ruedas binarias cuando es necesario y solicita un único reinicio limpio. Al
reconectar, use **Entorno de ejecución > Ejecutar todo** nuevamente.

In [ ]:
import os
import signal
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

PYTHON_OBJETIVO = (3, 13)
if sys.version_info[:2] != PYTHON_OBJETIVO:
    raise RuntimeError(
        f'Este notebook exige CPython 3.13; el runtime actual es '
        f'{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}.'
    )

VERSIONES = {
    'tensorflow': '2.21.0', 'pandas': '2.3.3',
    'scikit-learn': '1.6.1', 'matplotlib': '3.10.9',
    'numpy': '2.5.2', 'scipy': '1.18.1', 'joblib': '1.5.3',
}
MARCADOR_REINICIO = Path('/content/.phenological_stack_notebook_6_ready')
observadas = {}
for nombre in VERSIONES:
    try:
        observadas[nombre] = version(nombre)
    except PackageNotFoundError:
        observadas[nombre] = None
diferencias = {n: {'observada': observadas[n], 'esperada': v}
               for n, v in VERSIONES.items() if observadas[n] != v}
prueba_stack = None
if not diferencias:
    prueba_stack = subprocess.run([
        sys.executable, '-c',
        'import numpy, scipy, sklearn, pandas, matplotlib, joblib, tensorflow; '
        'print(numpy.__version__, scipy.__version__, sklearn.__version__)'
    ], capture_output=True, text=True)
requiere_instalacion = diferencias or (
    prueba_stack is not None and prueba_stack.returncode != 0
)
if requiere_instalacion:
    paquetes = [f'{n}=={v}' for n, v in VERSIONES.items()]
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
        '--force-reinstall', '--no-cache-dir', '--only-binary=:all:', *paquetes,
    ])
if requiere_instalacion or not MARCADOR_REINICIO.exists():
    MARCADOR_REINICIO.write_text('reinicio solicitado', encoding='utf-8')
    print('El kernel se reiniciará. Reconecte y ejecute todo nuevamente.')
    os.kill(os.getpid(), signal.SIGKILL)
for nombre, esperada in VERSIONES.items():
    observada = version(nombre)
    if observada != esperada:
        raise RuntimeError(f'{nombre}: esperado {esperada}; observado {observada}')
    print(f'OK {nombre}=={observada}')
print('Runtime compatible:', sys.version)


## 2. Datos públicos y directorio de ejecución

Los dos CSV están versionados en `data/` dentro del repositorio público. En
Colab se descargan desde GitHub y se verifica su contenido antes de leerlos.
La rama `main` actúa solo como localizador: los hashes fijados son el control
de identidad. No se usa Google Drive, credenciales ni rutas personales.

In [ ]:
import hashlib
import json
import platform
import random
import shutil
from datetime import datetime
from pathlib import Path
from urllib.request import urlretrieve

VERSION_NOTEBOOK = '0.1.0-notebook.6'
VERSION_FUENTE = '0.1.0-rc.3'
COMMIT_FUENTE = '253358e75ac6bf72333d358251613473f59961ee'
REPOSITORIO_NOTEBOOK = 'Lezier/phenological_prediction_notebook'
DATA_REF = 'main'
DATA_BASE_URL = f'https://raw.githubusercontent.com/{REPOSITORIO_NOTEBOOK}/{DATA_REF}/data'
EJECUTAR_COMPARACION_COMPLETA = True
VALORES_DEMO = [18.5, 24.0, 12.0, 15.0, 18.2, 62.0, 210.0]

RUN_ID = datetime.now().astimezone().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR_LOCAL = Path('/content') / f'phenological_prediction_run_{RUN_ID}'
DATA_DIR_LOCAL = RUN_DIR_LOCAL / 'data'
OUTPUT_DIR_LOCAL = RUN_DIR_LOCAL / 'output'
MODELS_DIR_LOCAL = RUN_DIR_LOCAL / 'models'
for carpeta in (DATA_DIR_LOCAL, OUTPUT_DIR_LOCAL, MODELS_DIR_LOCAL):
    carpeta.mkdir(parents=True, exist_ok=True)

HASHES_DATOS = {
    'base_fenologia_clima.csv': '0397C7A0B61B76388C22A1CDD1F13BCB2B7E10069C7BBB2935F0ADCC2E5CF6B7',
    'base_fenologia_clima_satelite.csv': '0C307E8CAAEEE04A87EB572AA675C4BB7C97EF1C1BC5C0D36C7018FB7129ADCA',
}
def sha256_texto_lf(ruta: Path) -> str:
    return hashlib.sha256(ruta.read_bytes().replace(b'\r\n', b'\n')).hexdigest().upper()
def sha256_raw(ruta: Path) -> str:
    return hashlib.sha256(ruta.read_bytes()).hexdigest().upper()

for nombre, esperado in HASHES_DATOS.items():
    destino = DATA_DIR_LOCAL / nombre
    url = f'{DATA_BASE_URL}/{nombre}'
    urlretrieve(url, destino)
    observado = sha256_texto_lf(destino)
    if observado != esperado:
        destino.unlink(missing_ok=True)
        raise ValueError(f'Hash incorrecto para {nombre}: {observado}; esperado {esperado}')
    print(f'OK {nombre}: {observado}')
print('Datos públicos descargados y verificados en:', DATA_DIR_LOCAL)


## 3. Configuración y funciones alineadas con RC3

Los folds se materializan una sola vez por escenario/validación y se reutilizan
en ambos clasificadores. Los pesos se calculan exclusivamente con `train` y se
aplican a la red y al bosque. El test externo no participa en `fit()`.

In [ ]:
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from time import perf_counter
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

CLIMA = ['clima_temp_media', 'clima_temp_max_media', 'clima_temp_min_media',
         'clima_precip_acumulada', 'clima_radiacion_media',
         'clima_humedad_media', 'clima_gdd_acumulado']
CLASIFICADORES = ('red_densa', 'random_forest')
FORMULA_PESO = 'n_train / (n_clases_presentes_train * frecuencia_clase_train)'
RF = {'n_estimators': 400, 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
RED = {
    'capas_ocultas': [{'unidades': 16, 'activacion': 'relu', 'dropout': 0.3},
                      {'unidades': 8, 'activacion': 'relu', 'dropout': 0.2}],
    'activacion_salida': 'softmax', 'learning_rate': 0.001,
    'epocas_maximas': 60, 'batch_size': 16,
    'early_stopping': {'monitor': 'loss', 'patience': 8, 'restore_best_weights': True},
}
TRATAMIENTO_EARLY_STOPPING = {
    'rol': 'control_de_convergencia_sobre_loss_de_train',
    'usa_validacion_interna': False, 'usa_fold_externo_test': False,
    'interpretable_como_control_de_overfitting': False,
    'decision_rc3': 'conservar_baseline_heredado_sin_cambiar_resultados',
}
CONFIGURACION = {
    'version_notebook': VERSION_NOTEBOOK, 'fuente_rc': VERSION_FUENTE,
    'commit_fuente': COMMIT_FUENTE, 'semilla': 42, 'folds': 5,
    'random_forest': RF, 'red_neuronal': RED,
    'modelos': {
        'a': ('base_fenologia_clima.csv', CLIMA, 'Modelo A - clima'),
        'aprima': ('base_fenologia_clima_satelite.csv', CLIMA, "Modelo A' - control clima"),
        'b': ('base_fenologia_clima_satelite.csv', CLIMA + ['ndvi'], 'Modelo B - clima + NDVI'),
    },
}

def fijar_semillas():
    random.seed(42); np.random.seed(42); tf.keras.utils.set_random_seed(42)
    try: tf.config.experimental.enable_op_determinism()
    except Exception as error: print('Aviso de determinismo:', error)

def cargar_datos(nombre):
    archivo, variables, titulo = CONFIGURACION['modelos'][nombre]
    datos = pd.read_csv(DATA_DIR_LOCAL / archivo)
    datos['_fila_fuente_csv'] = datos.index
    if nombre in {'aprima', 'b'}: datos = datos[datos['ndvi'].notna()].copy()
    requeridas = ([v for v in variables if v != 'clima_radiacion_media']
                  if nombre == 'a' else variables) + ['macro_etapa', 's_id']
    return datos.dropna(subset=requeridas).reset_index(drop=True), variables, titulo

def identificador_fold(validacion, numero, idx_train, idx_test):
    encabezado = json.dumps({'validacion': validacion, 'fold': numero,
        'semilla': 42, 'n_splits': 5}, sort_keys=True, separators=(',', ':')).encode()
    resumen = hashlib.sha256(encabezado)
    resumen.update(np.asarray(idx_train, dtype='<i8').tobytes()); resumen.update(b'|')
    resumen.update(np.asarray(idx_test, dtype='<i8').tobytes())
    return resumen.hexdigest().upper()

def construir_folds(y, grupos, validacion):
    indices = np.arange(len(y), dtype=np.int64)
    if validacion == 'aleatoria':
        partes = StratifiedKFold(5, shuffle=True, random_state=42).split(indices, y)
    else:
        partes = StratifiedGroupKFold(5, shuffle=True, random_state=42).split(indices, y, grupos)
    return [{'fold': n, 'fold_id': identificador_fold(validacion, n, tr, te),
             'train': np.asarray(tr), 'test': np.asarray(te)}
            for n, (tr, te) in enumerate(partes, 1)]

def pesos_train(y_train):
    clases = np.unique(y_train)
    valores = compute_class_weight('balanced', classes=clases, y=y_train)
    return {int(c): float(p) for c, p in zip(clases, valores)}

def crear_red(n_variables, n_clases):
    capas = [tf.keras.layers.Input(shape=(n_variables,))]
    for capa in RED['capas_ocultas']:
        capas += [tf.keras.layers.Dense(capa['unidades'], activation=capa['activacion']),
                  tf.keras.layers.Dropout(capa['dropout'])]
    capas += [tf.keras.layers.Dense(n_clases, activation=RED['activacion_salida'])]
    red = tf.keras.Sequential(capas)
    red.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=RED['learning_rate']),
                loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return red

def entrenar_red(x_train, y_train, x_test, n_clases, pesos):
    inicio = perf_counter(); escalador = StandardScaler()
    x_train = escalador.fit_transform(x_train)
    red = crear_red(x_train.shape[1], n_clases)
    parada = tf.keras.callbacks.EarlyStopping(**RED['early_stopping'])
    historial = red.fit(x_train, y_train, epochs=RED['epocas_maximas'],
        batch_size=RED['batch_size'], class_weight=pesos, callbacks=[parada], verbose=0)
    t_train = perf_counter() - inicio; inicio = perf_counter()
    pred = np.argmax(red(escalador.transform(x_test), training=False).numpy(), axis=1)
    t_pred = perf_counter() - inicio; epocas = len(historial.history['loss'])
    tf.keras.backend.clear_session()
    return pred, t_train, t_pred, epocas, epocas < RED['epocas_maximas']

def entrenar_bosque(x_train, y_train, x_test, pesos):
    inicio = perf_counter()
    bosque = RandomForestClassifier(**{**RF, 'class_weight': pesos}).fit(x_train, y_train)
    t_train = perf_counter() - inicio; inicio = perf_counter()
    pred = bosque.predict(x_test); t_pred = perf_counter() - inicio
    return pred, t_train, t_pred, None, None

def evaluar(nombre, validacion):
    datos, variables, titulo = cargar_datos(nombre)
    le = LabelEncoder(); y = le.fit_transform(datos['macro_etapa'])
    x = datos[variables].to_numpy(); grupos = datos['s_id'].to_numpy()
    folds = construir_folds(y, grupos, validacion); etiquetas = list(range(len(le.classes_)))
    matrices = {c: np.zeros((len(etiquetas), len(etiquetas)), dtype=int) for c in CLASIFICADORES}
    filas, asignaciones, evidencias_pesos = [], [], []
    for fold in folds:
        tr, te = fold['train'], fold['test']; pesos = pesos_train(y[tr])
        compartidas = set(grupos[tr]).intersection(grupos[te])
        if validacion == 'por_estacion' and compartidas: raise ValueError('Fuga entre estaciones')
        for particion, indices in (('train', tr), ('test', te)):
            for clasificador in CLASIFICADORES:
                for indice in indices:
                    asignaciones.append({'modelo_datos': nombre, 'validacion': validacion,
                        'clasificador': clasificador, 'fold': fold['fold'], 'fold_id': fold['fold_id'],
                        'particion': particion, 'fila_preparada': int(indice),
                        'fila_fuente_csv': int(datos.iloc[indice]['_fila_fuente_csv']),
                        'estacion': grupos[indice], 'clase': le.classes_[y[indice]]})
        clases_presentes, frecuencias = np.unique(y[tr], return_counts=True)
        frecuencias = dict(zip(clases_presentes, frecuencias))
        for clasificador in CLASIFICADORES:
            for codigo, clase in enumerate(le.classes_):
                frecuencia = int(frecuencias.get(codigo, 0))
                evidencias_pesos.append({'modelo_datos': nombre, 'validacion': validacion,
                    'clasificador': clasificador, 'fold': fold['fold'], 'fold_id': fold['fold_id'],
                    'n_train': len(tr), 'n_clases_presentes_train': len(clases_presentes),
                    'clase_codificada': codigo, 'clase': clase, 'frecuencia_train': frecuencia,
                    'peso_clase': pesos.get(codigo, np.nan), 'peso_aplicado': codigo in pesos,
                    'estrategia': 'balanced_explicito', 'formula': FORMULA_PESO,
                    'calculado_solo_con_train': True})
        imputador = SimpleImputer(strategy='median')
        x_train = imputador.fit_transform(x[tr]); x_test = imputador.transform(x[te])
        resultados = {
            'red_densa': entrenar_red(x_train, y[tr], x_test, len(etiquetas), pesos),
            'random_forest': entrenar_bosque(x_train, y[tr], x_test, pesos),
        }
        for clasificador, resultado in resultados.items():
            pred, t_train, t_pred, epocas, detencion = resultado
            matrices[clasificador] += confusion_matrix(y[te], pred, labels=etiquetas)
            presentes_train, presentes_test = set(y[tr]), set(y[te])
            filas.append({'modelo_datos': nombre, 'validacion': validacion,
                'clasificador': clasificador, 'fold': fold['fold'], 'fold_id': fold['fold_id'],
                'n_train': len(tr), 'n_test': len(te),
                'estaciones_train': len(set(grupos[tr])), 'estaciones_test': len(set(grupos[te])),
                'estaciones_compartidas': len(compartidas), 'clases_train': len(presentes_train),
                'clases_test': len(presentes_test),
                'clases_ausentes_train': '|'.join(le.classes_[i] for i in etiquetas if i not in presentes_train),
                'clases_ausentes_test': '|'.join(le.classes_[i] for i in etiquetas if i not in presentes_test),
                'accuracy': accuracy_score(y[te], pred),
                'f1_macro': f1_score(y[te], pred, labels=etiquetas, average='macro', zero_division=0),
                'f1_weighted': f1_score(y[te], pred, labels=etiquetas, average='weighted', zero_division=0),
                'tiempo_entrenamiento_segundos': t_train,
                'tiempo_inferencia_segundos': t_pred, 'tiempo_total_segundos': t_train + t_pred,
                'epocas_ejecutadas': epocas, 'early_stopping_detencion': detencion})
    for clasificador, matriz in matrices.items():
        tabla = pd.DataFrame(matriz, index=le.classes_, columns=le.classes_)
        tabla.index.name = 'clase_real'
        tabla.to_csv(OUTPUT_DIR_LOCAL / f'matriz_{nombre}_{validacion}_{clasificador}.csv')
    metadata = {'modelo_datos': nombre, 'titulo': titulo, 'validacion': validacion,
        'grupo': 's_id' if validacion == 'por_estacion' else None,
        'muestras': len(datos), 'estaciones': datos['s_id'].nunique(),
        'variables': variables, 'clases': list(le.classes_), 'folds_compartidos': True,
        'folds_ejecutados': [f['fold'] for f in folds],
        'tratamiento_early_stopping': TRATAMIENTO_EARLY_STOPPING}
    return (pd.DataFrame(filas), metadata, pd.DataFrame(asignaciones),
            pd.DataFrame(evidencias_pesos))

fijar_semillas()
for nombre in CONFIGURACION['modelos']:
    datos, variables, titulo = cargar_datos(nombre)
    print(f'{titulo}: {len(datos)} muestras, {datos.s_id.nunique()} estaciones, {len(variables)} variables')


## 4. Comparación completa

Se ejecutan 60 evaluaciones: 3 escenarios × 2 validaciones × 2 clasificadores
× 5 folds. Se exportan métricas, dispersión, tiempos, folds, pesos y matrices.
La validación agrupada por estación constituye la evidencia principal.

In [ ]:
if EJECUTAR_COMPARACION_COMPLETA:
    resultados, metadatos, asignaciones, ponderaciones = [], [], [], []
    for nombre in CONFIGURACION['modelos']:
        for validacion in ('aleatoria', 'por_estacion'):
            print(f'Ejecutando {nombre}: {validacion}')
            detalle_parcial, metadata, folds_csv, pesos_csv = evaluar(nombre, validacion)
            resultados.append(detalle_parcial); metadatos.append(metadata)
            asignaciones.append(folds_csv); ponderaciones.append(pesos_csv)
    detalle = pd.concat(resultados, ignore_index=True)
    asignacion_folds = pd.concat(asignaciones, ignore_index=True)
    pesos_folds = pd.concat(ponderaciones, ignore_index=True)
    resumen = detalle.groupby(['modelo_datos', 'validacion', 'clasificador'], as_index=False).agg(
        n_folds=('fold', 'nunique'), accuracy_promedio=('accuracy', 'mean'),
        accuracy_desviacion=('accuracy', 'std'), f1_macro_promedio=('f1_macro', 'mean'),
        f1_macro_desviacion=('f1_macro', 'std'),
        f1_weighted_promedio=('f1_weighted', 'mean'),
        f1_weighted_desviacion=('f1_weighted', 'std'),
        tiempo_entrenamiento_promedio_segundos=('tiempo_entrenamiento_segundos', 'mean'),
        tiempo_entrenamiento_desviacion_segundos=('tiempo_entrenamiento_segundos', 'std'),
        tiempo_inferencia_promedio_segundos=('tiempo_inferencia_segundos', 'mean'),
        tiempo_inferencia_desviacion_segundos=('tiempo_inferencia_segundos', 'std'),
        tiempo_total_promedio_segundos=('tiempo_total_segundos', 'mean'),
        tiempo_total_desviacion_segundos=('tiempo_total_segundos', 'std'))
    detalle.to_csv(OUTPUT_DIR_LOCAL / 'metricas_por_fold.csv', index=False)
    detalle[['modelo_datos','validacion','clasificador','fold','fold_id','n_train','n_test',
        'tiempo_entrenamiento_segundos','tiempo_inferencia_segundos','tiempo_total_segundos']].to_csv(
        OUTPUT_DIR_LOCAL / 'tiempos_por_fold.csv', index=False)
    asignacion_folds.to_csv(OUTPUT_DIR_LOCAL / 'asignacion_folds.csv', index=False)
    pesos_folds.to_csv(OUTPUT_DIR_LOCAL / 'pesos_clase_por_fold.csv', index=False)
    resumen.to_csv(OUTPUT_DIR_LOCAL / 'comparacion_consolidada.csv', index=False)

    etiquetas = [f'{f.modelo_datos}\n{f.validacion}\n{f.clasificador}' for f in resumen.itertuples()]
    posiciones = np.arange(len(resumen)); figura, eje = plt.subplots(figsize=(15, 6))
    eje.bar(posiciones-.18, resumen.accuracy_promedio, .36, label='Accuracy')
    eje.bar(posiciones+.18, resumen.f1_macro_promedio, .36, label='F1 macro')
    eje.set(title='Comparación RC3', ylabel='Métrica promedio', ylim=(0,1),
            xticks=posiciones, xticklabels=etiquetas)
    plt.setp(eje.get_xticklabels(), rotation=45, ha='right'); eje.legend(); figura.tight_layout()
    figura.savefig(OUTPUT_DIR_LOCAL / 'comparacion_metricas.png', dpi=160); plt.show()

    configuracion_salida = {**CONFIGURACION,
        'fecha_ejecucion': datetime.now().astimezone().isoformat(timespec='seconds'),
        'runtime': {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
            'tensorflow': tf.__version__, 'joblib': joblib.__version__},
        'parametros_random_forest_efectivos': RandomForestClassifier(**RF).get_params(False),
        'tratamiento_early_stopping': TRATAMIENTO_EARLY_STOPPING,
        'procedencia_hiperparametros': {
            'random_forest': 'heuristica_no_optimizada_incorporada_por_chatgpt',
            'red_neuronal': 'baseline_heredado_sin_procedencia_completamente_confirmada'},
        'hashes_datos_sha256_lf': HASHES_DATOS, 'metadatos_modelos': metadatos,
        'evidencia_folds': {'archivo': 'asignacion_folds.csv',
            'sha256': sha256_raw(OUTPUT_DIR_LOCAL/'asignacion_folds.csv')},
        'evidencia_ponderacion_clases': {'archivo': 'pesos_clase_por_fold.csv',
            'sha256': sha256_raw(OUTPUT_DIR_LOCAL/'pesos_clase_por_fold.csv'),
            'calculado_solo_con_train': True},
        'medicion_tiempos': {'archivo': 'tiempos_por_fold.csv',
            'reloj': 'time.perf_counter', 'unidad': 'segundos'}}
    (OUTPUT_DIR_LOCAL/'configuracion_ejecucion.json').write_text(
        json.dumps(configuracion_salida, ensure_ascii=False, indent=2), encoding='utf-8')

    referencia = {'accuracy_promedio': .8346299027206735,
        'accuracy_desviacion': .1372673420763116,
        'f1_macro_promedio': .7204350871824743,
        'f1_macro_desviacion': .114231889168188}
    observado = resumen.query("modelo_datos=='a' and validacion=='por_estacion' and clasificador=='random_forest'").iloc[0]
    verificacion = pd.DataFrame([{'metrica': m, 'referencia_rc3': v,
        'valor_colab': float(observado[m]), 'diferencia': float(observado[m])-v}
        for m,v in referencia.items()])
    verificacion.to_csv(OUTPUT_DIR_LOCAL/'verificacion_contra_rc3.csv', index=False)
    display(resumen); display(verificacion)
else:
    detalle = resumen = asignacion_folds = pesos_folds = None
    print('Comparación omitida: este modo no reproduce las métricas oficiales.')


## 5. Entrenamiento final de Random Forest A

El prototipo se ajusta con las 1.091 filas de A. Este entrenamiento no estima
desempeño: las métricas proceden de la validación cruzada anterior.

In [ ]:
datos_a, variables_a, _ = cargar_datos('a')
if len(datos_a) != 1091 or datos_a['s_id'].nunique() != 41:
    raise ValueError('Modelo A debe contener 1.091 muestras y 41 estaciones.')
modelo_final = Pipeline([('imputador', SimpleImputer(strategy='median')),
    ('clasificador', RandomForestClassifier(**RF))])
modelo_final.fit(datos_a[variables_a], datos_a['macro_etapa'].astype(str))
rangos = {v: {'min': float(datos_a[v].min()), 'max': float(datos_a[v].max())}
          for v in variables_a}
ADVERTENCIA = ('Uso experimental con datos europeos; no validado para operación en Chile '
               'y no sustituye evaluación agronómica. Las probabilidades no están calibradas.')
paquete = {'version_esquema': 2, 'version_proyecto': VERSION_FUENTE,
    'commit_fuente': COMMIT_FUENTE, 'pipeline': modelo_final, 'variables': variables_a,
    'clases': list(modelo_final.named_steps['clasificador'].classes_),
    'rangos_entrenamiento': rangos, 'sha256_datos': HASHES_DATOS['base_fenologia_clima.csv'],
    'probabilidades_calibradas': False, 'advertencia': ADVERTENCIA}
ruta_modelo = MODELS_DIR_LOCAL/'random_forest_a_colab.joblib'
joblib.dump(paquete, ruta_modelo, compress=3); hash_modelo = sha256_raw(ruta_modelo)
metadata_final = {'version_notebook': VERSION_NOTEBOOK, 'fuente_rc': VERSION_FUENTE,
    'commit_fuente': COMMIT_FUENTE,
    'fecha_entrenamiento': datetime.now().astimezone().isoformat(timespec='seconds'),
    'modelo': 'Random Forest A - clima', 'muestras_entrenamiento': 1091,
    'estaciones': 41, 'variables': variables_a, 'clases': paquete['clases'],
    'parametros_random_forest_explicitos': RF,
    'parametros_random_forest_efectivos': RandomForestClassifier(**RF).get_params(False),
    'sha256_datos_lf': HASHES_DATOS['base_fenologia_clima.csv'],
    'sha256_modelo_colab': hash_modelo, 'probabilidades_calibradas': False,
    'nota_hash': 'La serialización depende del entorno y no sustituye el modelo Python RC3.',
    'advertencia': ADVERTENCIA}
(OUTPUT_DIR_LOCAL/'metadata_modelo_colab.json').write_text(
    json.dumps(metadata_final, ensure_ascii=False, indent=2), encoding='utf-8')
print('Modelo Colab generado:', ruta_modelo); print('SHA-256:', hash_modelo)


## 6. Inferencia demostrativa

La demo recibe siete valores en el orden documentado, devuelve una etapa y
cinco puntajes no calibrados, y advierte valores fuera del rango observado.

In [ ]:
def ejecutar_demo(paquete_modelo, valores):
    if len(valores) != len(paquete_modelo['variables']):
        raise ValueError('La demo requiere exactamente siete valores.')
    arreglo = np.asarray(valores, dtype=float)
    if not np.isfinite(arreglo).all(): raise ValueError('Todos los valores deben ser finitos.')
    fuera = [v for v,x in zip(paquete_modelo['variables'], arreglo)
             if x < paquete_modelo['rangos_entrenamiento'][v]['min']
             or x > paquete_modelo['rangos_entrenamiento'][v]['max']]
    entrada = pd.DataFrame([arreglo], columns=paquete_modelo['variables'])
    modelo = paquete_modelo['pipeline']; clase = str(modelo.predict(entrada)[0])
    probabilidades = modelo.predict_proba(entrada)[0]
    resultado = {'entrada': dict(zip(paquete_modelo['variables'], arreglo.tolist())),
        'macro_etapa_estimada': clase,
        'probabilidades_no_calibradas': dict(zip(modelo.classes_, map(float, probabilidades))),
        'probabilidades_calibradas': False, 'variables_fuera_rango': fuera,
        'advertencia': paquete_modelo['advertencia']}
    print('Macro-etapa estimada:', clase); print('Probabilidades estimadas (no calibradas):')
    for etiqueta, p in sorted(resultado['probabilidades_no_calibradas'].items(),
                              key=lambda e:e[1], reverse=True): print(f'  - {etiqueta}: {p:.1%}')
    if fuera: print('Advertencia de rango:', ', '.join(fuera))
    print('Advertencia:', paquete_modelo['advertencia']); return resultado

resultado_demo = ejecutar_demo(paquete, VALORES_DEMO)
(OUTPUT_DIR_LOCAL/'resultado_demo.json').write_text(
    json.dumps(resultado_demo, ensure_ascii=False, indent=2), encoding='utf-8')


## 7. Controles y manifiesto de la ejecución

La última celda comprueba contratos esenciales y genera un manifiesto local.
CP15 incorporará el empaquetado ZIP y su descarga directa desde Colab.

In [ ]:
for nombre, esperado in HASHES_DATOS.items():
    assert sha256_texto_lf(DATA_DIR_LOCAL/nombre) == esperado
if EJECUTAR_COMPARACION_COMPLETA:
    assert len(detalle) == 60 and len(resumen) == 12
    assert (resumen['n_folds'] == 5).all()
    assert (detalle.query("validacion=='por_estacion'")['estaciones_compartidas'] == 0).all()
    assert not detalle[['accuracy','f1_macro','f1_weighted']].isna().any().any()
    assert len(asignacion_folds) == 48100 and len(pesos_folds) == 300
    claves = ['modelo_datos','validacion','fold','fold_id']
    for _, grupo in asignacion_folds.groupby(claves):
        assert set(grupo['clasificador']) == set(CLASIFICADORES)
    assert pesos_folds['calculado_solo_con_train'].all()
    for columna in ['tiempo_entrenamiento_segundos','tiempo_inferencia_segundos','tiempo_total_segundos']:
        assert (detalle[columna] >= 0).all()

archivos_exportar = [*OUTPUT_DIR_LOCAL.glob('*'), *MODELS_DIR_LOCAL.glob('*')]
manifiesto = {'run_id': RUN_ID,
    'fecha_exportacion': datetime.now().astimezone().isoformat(timespec='seconds'),
    'version_notebook': VERSION_NOTEBOOK, 'fuente_rc': VERSION_FUENTE,
    'commit_fuente': COMMIT_FUENTE, 'comparacion_completa': EJECUTAR_COMPARACION_COMPLETA,
    'archivos': [{'ruta': str(r.relative_to(RUN_DIR_LOCAL)), 'sha256_raw': sha256_raw(r)}
                 for r in archivos_exportar if r.is_file()]}
(RUN_DIR_LOCAL/'manifest_ejecucion.json').write_text(
    json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding='utf-8')
print('CONTROLES APROBADOS')
print('Resultados locales:', RUN_DIR_LOCAL)
print('CP15 añadirá la descarga ZIP de esta carpeta.')
